# 04 — Classical Linear Codes and the Hamming Code

## Read before you begin

- Arthur Pesah, [*Classical Error Correction*](https://arthurpesah.me/blog/2022-05-21-classical-error-correction/).
- Sections to read: **General setting**, **Distance and $[n,k,d]$ codes**, **Parity-checks and Hamming codes**, **Linear codes through syndrome decoding**.
- Concepts to extract before starting: what makes a code *linear*; the $[n,k,d]$ notation (block length, message length, minimum distance); how a generator matrix $G$ and a parity-check matrix $H$ describe the same code from two directions; why the syndrome $s = He$ depends only on the error, not the transmitted codeword.

## Learning objectives

This notebook consolidates that reading by building the standard $[7,4,3]$ Hamming code from scratch. By the end you should be able to:

1. compute Hamming weight and Hamming distance, and state why minimum distance sets a code's error-correcting power;
2. build a generator matrix $G$ and parity-check matrix $H$ for the $[7,4,3]$ Hamming code and verify $HG = 0 \pmod 2$;
3. show codewords are exactly the null space of $H$, and explain why the syndrome $s = He$ isolates the error;
4. decode a received word by looking up its syndrome, and verify this for every possible single-bit error;
5. verify, by enumeration, that the code's minimum distance really is 3;
6. compare the Hamming code's efficiency to the $[3,1,3]$ repetition code from notebook 02;
7. state why this matters for the rest of the roadmap: classical parity checks are about to become quantum stabilizer checks (notebook 05 builds the Steane code directly from this $H$).

**Expected outputs:** a working `encode`/`decode` pair for the $[7,4,3]$ Hamming code, a table of all 7 single-bit-error syndromes, and two independent numerical confirmations that the code's distance is exactly 3.

**Assumes:** notebook 02 (Hamming weight/distance are direct generalizations of the bit-flip counting used there) and notebook 03 (stabilizers as generalized parity checks).

**Tooling note:** all linear algebra in this notebook is done over $\mathrm{GF}(2)$ (bits, with addition = XOR) using only the Python standard library — no NumPy. Vectors are plain tuples of `0`/`1`, and matrices are tuples of tuples.

## 1. Hamming weight and Hamming distance

The **Hamming weight** of a bit vector is its number of 1s. The **Hamming distance** between two bit vectors of the same length is the number of positions where they differ — equivalently, the weight of their bitwise XOR. This is exactly the "how many bits flipped" question notebook 02 already answered informally for 3-bit codewords; here it becomes a named, general tool.

In [1]:
def weight(v):
    """Hamming weight: the number of 1s in a bit vector (tuple of 0/1)."""
    return sum(v)


def xor_vec(u, v):
    """Bitwise XOR of two equal-length bit vectors."""
    return tuple(a ^ b for a, b in zip(u, v))


def distance(u, v):
    """Hamming distance: the number of positions where u and v differ."""
    return weight(xor_vec(u, v))


# Sanity check against notebook 02's 3-bit repetition codewords.
print("weight(000) =", weight((0, 0, 0)))
print("weight(111) =", weight((1, 1, 1)))
print("distance(000, 111) =", distance((0, 0, 0), (1, 1, 1)))
print("distance(000, 011) =", distance((0, 0, 0), (0, 1, 1)))

weight(000) = 0
weight(111) = 3
distance(000, 111) = 3
distance(000, 011) = 2


## 2. $[n, k, d]$ codes and why distance matters

A code is described by three numbers, $[n, k, d]$:

- $n$ — the **block length**: how many bits are transmitted per codeword.
- $k$ — the **message length**: how many bits of actual information each codeword carries.
- $d$ — the **minimum distance**: the smallest Hamming distance between any two *distinct* codewords.

Minimum distance sets the code's guarantees, exactly as in notebook 02:

- it **detects** up to $d - 1$ errors,
- it **corrects** up to $\lfloor (d-1)/2 \rfloor$ errors.

The 3-bit repetition code from notebook 02 is a $[3, 1, 3]$ code: 3 bits transmitted, 1 bit of message, minimum distance 3 (`000` and `111` differ in all 3 positions). This notebook builds a $[7, 4, 3]$ code — same distance, hence the same single-error correction guarantee, but carrying 4 message bits instead of 1. Section 7 makes that efficiency gain concrete.

### Prediction 1

**Before running the next section:** a $[7,4,3]$ code corrects up to $\lfloor (3-1)/2 \rfloor = 1$ error, same as the repetition code. Given it carries 4 message bits in 7 transmitted bits instead of 1 message bit in 3, do you expect it to need *more*, *fewer*, or the *same number* of redundant (non-message) bits per message bit as the repetition code? Work out the ratio for each before continuing.

## 3. Linear codes: generator matrix $G$ and parity-check matrix $H$

A code is **linear** if every codeword is a linear combination (over $\mathrm{GF}(2)$, i.e. XOR) of a fixed set of basis codewords. That lets us describe the whole code with two matrices instead of listing all $2^k$ codewords:

- The **generator matrix** $G$ ($n \times k$) turns a message into a codeword: $c = Gm \pmod 2$, for a message column vector $m \in \{0,1\}^k$.
- The **parity-check matrix** $H$ ($(n-k) \times n$) tests whether a received word is a valid codeword: $Hc = 0 \pmod 2$ if and only if $c$ is a codeword. Equivalently, **the set of codewords is exactly the null space of $H$**.

Because $c = Gm$ must satisfy $Hc = 0$ for *every* message $m$, $G$ and $H$ must satisfy $HG = 0 \pmod 2$ — this is the direct algebraic link between the two descriptions, and it's the first thing we'll verify once both matrices are built.

### The $[7,4,3]$ Hamming code's parity-check matrix

The classic construction places $H$'s columns in numerical order: column $j$ (for $j = 1, \dots, 7$) is the 3-bit binary representation of $j$ itself.

In [2]:
def bits_of(j, width):
    """j as a tuple of `width` bits, most-significant bit first."""
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))


# H's columns, in order, are the binary representations of 1..7.
H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = tuple(tuple(col[row] for col in H_columns) for row in range(3))

print("H =")
for row in H:
    print(row)

H =
(0, 0, 0, 1, 1, 1, 1)
(0, 1, 1, 0, 0, 1, 1)
(1, 0, 1, 0, 1, 0, 1)


Positions 1, 2, and 4 are powers of two, so their columns are the standard basis vectors $(0,0,1)$, $(0,1,0)$, $(1,0,0)$ — each affects exactly one row of $H$. These become the **parity bits**. The remaining positions (3, 5, 6, 7) become the **data bits**: each parity bit is defined as the XOR of the data bits whose columns share a 1 in that row.

In [3]:
def matvec_gf2(matrix, vec):
    """Matrix-vector product mod 2: rows of `matrix` dotted with `vec`."""
    return tuple(sum(a * b for a, b in zip(row, vec)) % 2 for row in matrix)


def encode(message):
    """Encode a 4-bit message (m1, m2, m3, m4) into a 7-bit Hamming codeword.

    Codeword positions are 1-indexed: 1, 2, 4 are parity bits; 3, 5, 6, 7 carry
    the message bits m1, m2, m3, m4 respectively. This placement is exactly why
    H's columns being 1..7 makes each parity bit a plain XOR of specific data bits.
    """
    m1, m2, m3, m4 = message
    p1 = m1 ^ m2 ^ m4  # row "bit_low" of H: positions 1, 3, 5, 7
    p2 = m1 ^ m3 ^ m4  # row "bit_mid" of H: positions 2, 3, 6, 7
    p4 = m2 ^ m3 ^ m4  # row "bit_high" of H: positions 4, 5, 6, 7
    # 1-indexed layout (1..7): p1, p2, m1, p4, m2, m3, m4
    return (p1, p2, m1, p4, m2, m3, m4)


for message in [(0, 0, 0, 0), (1, 0, 0, 0), (0, 1, 0, 0), (1, 1, 0, 1)]:
    print(f"encode({message}) = {encode(message)}")

encode((0, 0, 0, 0)) = (0, 0, 0, 0, 0, 0, 0)
encode((1, 0, 0, 0)) = (1, 1, 1, 0, 0, 0, 0)
encode((0, 1, 0, 0)) = (1, 0, 0, 1, 1, 0, 0)
encode((1, 1, 0, 1)) = (1, 0, 1, 0, 1, 0, 1)


In [4]:
def matmul_gf2(A, B):
    """Matrix product mod 2. A is m x n, B is n x p (both tuples of tuples)."""
    m, n, p = len(A), len(A[0]), len(B[0])
    result = [[0] * p for _ in range(m)]
    for i in range(m):
        for k in range(n):
            if A[i][k]:
                for j in range(p):
                    result[i][j] ^= B[k][j]
    return tuple(tuple(row) for row in result)


# Build G by encoding each standard basis message -- G's columns are exactly
# encode(e1), encode(e2), encode(e3), encode(e4).
basis_messages = [(1, 0, 0, 0), (0, 1, 0, 0), (0, 0, 1, 0), (0, 0, 0, 1)]
G_columns = [encode(m) for m in basis_messages]
G = tuple(tuple(col[row] for col in G_columns) for row in range(7))

print("G =")
for row in G:
    print(row)

G =
(1, 1, 0, 1)
(1, 0, 1, 1)
(1, 0, 0, 0)
(0, 1, 1, 1)
(0, 1, 0, 0)
(0, 0, 1, 0)
(0, 0, 0, 1)


### Verification: $HG = 0 \pmod 2$

In [5]:
HG = matmul_gf2(H, G)
print("HG =")
for row in HG:
    print(row)

print("HG is the all-zero 3x4 matrix:", all(v == 0 for row in HG for v in row))

HG =
(0, 0, 0, 0)
(0, 0, 0, 0)
(0, 0, 0, 0)
HG is the all-zero 3x4 matrix: True


## 4. Codewords are the null space of $H$

Every one of the $2^4 = 16$ possible messages produces a codeword, and every one of those codewords should satisfy $Hc = 0$. This is exactly notebook 03's "code space = simultaneous $+1$ eigenspace of the stabilizers" idea, translated into classical linear algebra: **the code space is the null space of the check matrix.**

In [6]:
from itertools import product

all_messages = list(product([0, 1], repeat=4))
codewords = [encode(m) for m in all_messages]

all_valid = all(matvec_gf2(H, c) == (0, 0, 0) for c in codewords)
print(f"Generated {len(codewords)} codewords; all satisfy Hc = 0: {all_valid}")
print("First 4 codewords:", codewords[:4])

Generated 16 codewords; all satisfy Hc = 0: True
First 4 codewords: [(0, 0, 0, 0, 0, 0, 0), (1, 1, 0, 1, 0, 0, 1), (0, 1, 0, 1, 0, 1, 0), (1, 0, 0, 0, 0, 1, 1)]


## 5. The syndrome $s = He$

If a codeword $c$ is corrupted by an error $e$ (a bit vector marking which positions flipped), the received word is $r = c \oplus e$. Because $H$ is linear over $\mathrm{GF}(2)$:

$$Hr = H(c \oplus e) = Hc \oplus He = 0 \oplus He = He.$$

The syndrome depends **only on the error**, never on which codeword was sent — precisely the classical analogue of notebook 03's claim that a stabilizer measurement reveals nothing about the encoded logical value.

### Prediction 2

Recall that $H$'s columns are, in order, the binary representations of $1, 2, \dots, 7$. **Before running the next cell:** if a single-bit error flips position $j$ (1-indexed), what do you expect the syndrome $He$ to equal, written as a 3-bit number? (Hint: $e$ is the unit vector with a 1 only at position $j$ — what does $H$ do to a unit vector?)

In [7]:
def unit_error(position, n=7):
    """A length-n error vector with a single 1 at the given 1-indexed position."""
    return tuple(1 if i == position else 0 for i in range(1, n + 1))


def syndrome(H, received_or_error):
    return matvec_gf2(H, received_or_error)


def bits_to_int(bits):
    value = 0
    for b in bits:
        value = (value << 1) | b
    return value


print(f"{'position j':>10} {'error vector':>20} {'syndrome':>10} {'as integer':>10}")
for j in range(1, 8):
    e = unit_error(j)
    s = syndrome(H, e)
    print(f"{j:>10} {str(e):>20} {str(s):>10} {bits_to_int(s):>10}")

position j         error vector   syndrome as integer
         1 (1, 0, 0, 0, 0, 0, 0)  (0, 0, 1)          1
         2 (0, 1, 0, 0, 0, 0, 0)  (0, 1, 0)          2
         3 (0, 0, 1, 0, 0, 0, 0)  (0, 1, 1)          3
         4 (0, 0, 0, 1, 0, 0, 0)  (1, 0, 0)          4
         5 (0, 0, 0, 0, 1, 0, 0)  (1, 0, 1)          5
         6 (0, 0, 0, 0, 0, 1, 0)  (1, 1, 0)          6
         7 (0, 0, 0, 0, 0, 0, 1)  (1, 1, 1)          7


As predicted: the syndrome of a single-bit error at position $j$ is exactly the binary representation of $j$. This is not a coincidence — it is *why* $H$'s columns were chosen to be $1, \dots, 7$ in order: the syndrome directly names the error location, with no separate lookup table needed (contrast this with notebook 02's `SYNDROME_TABLE`, which had to be built by hand for the repetition code).

## 6. Decoding

Decoding now has a direct rule: compute the syndrome; if it is nonzero, interpret it as an integer $j$ and flip the received bit at position $j$; then read off the message bits from positions 3, 5, 6, 7.

In [8]:
def flip_at(word, position):
    """Flip the bit at 1-indexed `position` in `word`."""
    bits = list(word)
    bits[position - 1] ^= 1
    return tuple(bits)


def decode(received):
    """Correct up to one bit flip, then extract the 4-bit message."""
    s = syndrome(H, received)
    error_position = bits_to_int(s)
    corrected = received if error_position == 0 else flip_at(received, error_position)
    # 1-indexed data positions are 3, 5, 6, 7 -> 0-indexed 2, 4, 5, 6.
    return (corrected[2], corrected[4], corrected[5], corrected[6])


message = (1, 0, 1, 1)
clean = encode(message)
for flip_position in [None, 1, 2, 3, 4, 5, 6, 7]:
    received = clean if flip_position is None else flip_at(clean, flip_position)
    decoded = decode(received)
    label = "no error" if flip_position is None else f"flip bit {flip_position}"
    print(f"{label:<10} received={received} decoded={decoded} correct={decoded == message}")

no error   received=(0, 1, 1, 0, 0, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 1 received=(1, 1, 1, 0, 0, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 2 received=(0, 0, 1, 0, 0, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 3 received=(0, 1, 0, 0, 0, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 4 received=(0, 1, 1, 1, 0, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 5 received=(0, 1, 1, 0, 1, 1, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 6 received=(0, 1, 1, 0, 0, 0, 1) decoded=(1, 0, 1, 1) correct=True
flip bit 7 received=(0, 1, 1, 0, 0, 1, 0) decoded=(1, 0, 1, 1) correct=True


Every single-bit error, on any of the 7 positions, decodes back to the original 4-bit message. This is the $[7,4,3]$ code doing exactly what a minimum distance of 3 promises.

## 7. Verifying the distance is really 3

There are two independent ways to check this by brute force, since there are only $2^4 = 16$ codewords.

**Method 1 — minimum weight.** For a *linear* code, the minimum distance between any two codewords equals the minimum Hamming weight among the *nonzero* codewords (because the difference of two codewords is itself a codeword). This is a useful shortcut, but let's not just trust it — Method 2 checks it directly.

In [9]:
nonzero_codewords = [c for c in codewords if weight(c) > 0]
min_weight = min(weight(c) for c in nonzero_codewords)
print("Minimum nonzero codeword weight:", min_weight)

Minimum nonzero codeword weight: 3


**Method 2 — brute-force pairwise distance.** Check every one of the $\binom{16}{2} = 120$ pairs of distinct codewords directly, with no shortcut.

In [10]:
from itertools import combinations

pairwise_distances = [distance(a, b) for a, b in combinations(codewords, 2)]
min_pairwise_distance = min(pairwise_distances)

print("Minimum pairwise distance over all 120 pairs:", min_pairwise_distance)
print("Both methods agree on distance 3:", min_weight == min_pairwise_distance == 3)

Minimum pairwise distance over all 120 pairs: 3
Both methods agree on distance 3: True


### Prediction 3

**Before running the next cell:** the $[3,1,3]$ repetition code uses 3 transmitted bits per 1 message bit (2 redundant bits per message bit). The $[7,4,3]$ Hamming code uses 7 transmitted bits per 4 message bits. Compute, by hand, the number of redundant bits *per message bit* for the Hamming code. Is it more or less redundant than the repetition code, given that both correct exactly 1 error?

In [11]:
def redundancy_per_message_bit(n, k):
    return (n - k) / k


repetition_redundancy = redundancy_per_message_bit(n=3, k=1)
hamming_redundancy = redundancy_per_message_bit(n=7, k=4)

print(f"[3,1,3] repetition code: {repetition_redundancy:.3f} redundant bits per message bit")
print(f"[7,4,3] Hamming code:    {hamming_redundancy:.3f} redundant bits per message bit")
print(f"Hamming code rate k/n = {4/7:.3f} vs. repetition code rate k/n = {1/3:.3f}")

[3,1,3] repetition code: 2.000 redundant bits per message bit
[7,4,3] Hamming code:    0.750 redundant bits per message bit
Hamming code rate k/n = 0.571 vs. repetition code rate k/n = 0.333


Both codes correct exactly one error (distance 3), but the Hamming code needs far less redundancy to do it: 0.75 extra bits per message bit, versus 2 for the repetition code. Spreading parity checks over more data bits, instead of just copying one bit three times, is a strictly better use of redundancy — the central reason coding theory moved past simple repetition.

## 8. From classical parity checks to quantum stabilizer checks

Every idea in this notebook has a name from notebook 03 already attached to it:

| This notebook (classical) | Notebook 03 (quantum) |
|---|---|
| Parity-check matrix $H$ | A list of stabilizer generators |
| A row of $H$ (one parity check) | One stabilizer, e.g. $Z_0Z_1$ |
| Codewords = null space of $H$ | Code space = simultaneous $+1$ eigenspace of the stabilizers |
| Syndrome $s = He$ | Syndrome (vector of measured stabilizer eigenvalues, as bits) |
| Syndrome depends only on the error | Stabilizer measurement reveals nothing about the encoded logical value |

The next notebook makes this correspondence literal rather than analogical: the **Steane code** is built by turning this exact $H$ matrix into *two* sets of quantum stabilizers (one built from $X$s, one from $Z$s) using the **CSS construction** — a general recipe for building a quantum code directly out of a classical linear code like this one.

## Exercises

**1.** Write `decode_two_errors_demo()` that applies `decode` to a received word with **two** bit flips (e.g. flip positions 1 and 2 of a clean codeword) and prints the result. Is the decoded message still correct? Relate your answer to the distance-3 correction bound from Section 2 — this code was never guaranteed to survive 2 errors, and here is what that failure actually looks like.

**2.** Write `is_linear_code(codewords)` that checks the defining property of a linear code directly: for every pair of codewords $c_1, c_2$ in the list, their XOR (`xor_vec`) is also in the list. Run it on the 16 Hamming codewords from Section 4 to confirm the code actually is linear (this property is what let Section 3 describe all 16 codewords with just $G$ and $H$ instead of listing them by hand).

**3.** The construction in Section 3 fixed $H$'s column order to $1, \dots, 7$. Build a second parity-check matrix `H_shuffled` with the same 7 columns in a different order (still all of $\{1,\dots,7\}$'s binary representations, just permuted), derive its own `G_shuffled` the same way Section 3 did, and check whether the resulting code still has distance 3. (It should — permuting bit positions doesn't change any pairwise distance — but the syndrome-equals-error-position shortcut from Section 5 will no longer hold in the same simple form. Confirm that too.)

In [12]:
def decode_two_errors_demo():
    pass  # TODO: implement, then remove this stub body


def is_linear_code(codewords):
    pass  # TODO: implement, then remove this stub body


# TODO: once implemented, try e.g.:
# decode_two_errors_demo()
# print(is_linear_code(codewords))